# Volatility Environment Phase 1
固定28戦略・16,298 tradesの市場環境診断。Entry除外0、EA/live変更なし。
Primary: **D1 ATR20 = TRの20日単純平均**（既存ATR実装優先）。Wilder ATRではありません。
Robustness: 20 log returnsのsample std×sqrt(252)。直前完了日をそれ以前252日と比較、midrank三分位。
計画: cb36e1c7d5403c9be40480cdcfbaf221b32fda00。
H1 ATR14 P70 Entry filter（通称ATR70）とは別研究。2022–2026は既閲覧です。

下のRESEARCH_SHAは、このNotebookを含む実装コミットの完全SHAを指定します。実装前計画コミットを指定してはいけません。
Driveは既存データ読込に必要な場合だけマウント。結果のDrive保存は最後のセルで初期OFF。


In [ ]:
from pathlib import Path
RESEARCH_SHA = "REPLACE_WITH_VERIFIED_IMPLEMENTATION_SHA"
BASELINE = Path("/content/drive/MyDrive/time-entry-portfolio-lab/daily_stop/baseline_cc32f32e3df5/daily_stop_baseline_trades.csv")
M1_ROOT = Path("/content/drive/MyDrive")
OUT = Path("/content")
MOUNT_INPUT_DRIVE = True
if MOUNT_INPUT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
import urllib.request, sys, subprocess
assert len(RESEARCH_SHA)==40 and all(c in '0123456789abcdef' for c in RESEARCH_SHA)
base = f"https://raw.githubusercontent.com/TR-KJ/time-entry-portfolio-lab/{RESEARCH_SHA}/"
stage = Path('/content/volatility_phase1_code'); stage.mkdir(exist_ok=True)
for filename in ['src/research/volatility_phase1.py','src/research/volatility_phase1_frozen_inputs.json','tests/verify_volatility_phase1.py','tests/test_volatility_phase1.py']:
    (stage/Path(filename).name).write_bytes(urllib.request.urlopen(base+filename).read())
sys.path.insert(0,str(stage))
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(stage),'-v'],check=True)
from volatility_phase1 import run
result = run(BASELINE,M1_ROOT,OUT,RESEARCH_SHA)


In [ ]:
from IPython.display import display
for name in ['combined_decision','decision','group_summary','strategy_primary','strategy_robustness']:
    print(name); display(result[name])
coverage=result['regime_coverage']
display(coverage[(coverage.Period=='FULL') & (coverage.Scope=='Portfolio')])
display(result['verification']); display(result['manual_audit'])


## 解釈上の範囲
HIGH−LOW差が主比較で、NORMALだけ異なる依存は検出できません。
CIは5000回の暦週cluster bootstrap（95%、seed20260913）。週内の戦略間依存は維持しますが、週を跨ぐ自己相関や非定常性は完全には扱いません。多重比較未調整の探索的診断です。
この結果でfilter/live採用を決めません。Phase 2は本結果確定後の別研究・別事前登録です。


In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    import shutil
    target = Path('/content/drive/MyDrive/time-entry-portfolio-lab/volatility_phase1')
    target.mkdir(parents=True,exist_ok=True)
    for p in OUT.glob('volatility_phase1_*.csv'): shutil.copy2(p,target/p.name)
